# ViSta — a worked example on simulated observations

ViSta stacks interferometric observations of many faint galaxies **in the uv
plane** and measures the stacked emission line and continuum: profile, flux,
width, size and significance.

This notebook runs it on a **simulated sample**: twelve ALMA Band 3
observations of galaxies at z = 2–3 whose CO(3–2) line is too faint to be
measured in any one of them. Every dataset is simulated twice, **with and
without noise**, so each step can be compared with the signal as it is, and
the true values are drawn in every figure (green, dashed).

1. the simulated sample;
2. one galaxy, without and with noise, in the uv plane and in the image;
3. stacking, with the continuum subtraction and the compression in one pass;
4. measuring the stacked line at a known width (`detect`);
5. searching for it, centre and width free (`search`);
6. what each option changes;
7. the continuum;
8. redoing the continuum subtraction without stacking again;
9. the stack in the image, without and with noise;
10. the same from the command line, and on your own data.

On Colab: *Runtime → Run all* (about 15 minutes on the CPU). Every
parameter is in the code cells: change it and run again from that cell.
The figures are also saved in `figures/`.

## 0. Setup

The code is cloned from GitHub, the C++/OpenMP kernel (`ms_ops`) is built
with CMake, and CASA's `casatasks` and `casatools` are installed for the
images. For a private repository, add a GitHub token to the Colab secrets
(key icon on the left) as `GITHUB_TOKEN`; or upload a zip of the repository
and unzip it in `/content`: the cell then uses it. On Colab the first import
of CASA downloads its data tables (a few minutes).

In [ ]:
REPO_URL = "https://github.com/martitors/ViSta-HPC.git"
BRANCH = "extract_opt"

import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO = os.environ.get("VISTA_REPO")            # a local checkout, to run this outside Colab
if REPO is None:
    REPO = "/content/ViSta-HPC" if IN_COLAB else os.path.abspath("ViSta-HPC")
    if not os.path.isdir(REPO):
        url = REPO_URL
        if IN_COLAB:
            try:                                 # private repository: token from the Colab secrets
                from google.colab import userdata
                url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
            except Exception:
                pass
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
print("ViSta source:", REPO)


def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(cmd)


if IN_COLAB:
    sh(f"pip install -q -r {REPO}/requirements.txt pybind11 casatasks casatools")
if not glob.glob(f"{REPO}/vista/ms_ops*.so"):          # build the kernel once
    sh(f"cmake -S {REPO} -B {REPO}/build -DCMAKE_BUILD_TYPE=Release > /dev/null"
       f" && cmake --build {REPO}/build -j 2 > /dev/null && cmake --install {REPO}/build > /dev/null")

sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "examples"))   # simulate.py, plot_line.py
import vista
from vista import ms_ops                               # the compiled kernel
print("ViSta", vista.__version__, "· kernel", os.path.basename(ms_ops.__file__))

In [ ]:
import json, math, time, shutil, logging, contextlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse
from daskms.experimental.zarr import xds_from_zarr
from casatasks import tclean
from casatools import image as iatool

import simulate                                        # the simulated observations
import plot_line as pl                                 # the figures of a line measurement
from vista import ViSta, export_ms
from vista.extract import (LineConfig, ContinuumConfig, BinningConfig, WeightingConfig,
                           LineFitConfig, BootstrapConfig, ProfileConfig, UVFitConfig,
                           measure_line, extract_flux, subtract_continuum, compress_visibilities)

logging.getLogger("daskms").setLevel(logging.ERROR)    # quiet about empty optional columns
WORK = "/content/vista_demo" if IN_COLAB else os.path.abspath("vista_demo")
os.makedirs(f"{WORK}/figures", exist_ok=True)
os.chdir(WORK)
pl.apply_style()                                       # one style for every figure
plt.rcParams.update({"figure.dpi": 110})
C_KMS = 299792.458

TRUE = dict(color="#1e8449", ls="--", lw=1.8)          # the true signal
CLEAN = dict(color="#2471a3", lw=1.3)                  # the data without noise
NOISY = dict(color="k", lw=0.9)                        # the data with noise


def save(fig, name):
    fig.savefig(f"figures/{name}.pdf", bbox_inches="tight")
    plt.show()

## 1. The simulated sample

`examples/simulate.py` writes Measurement Sets of an ALMA-like array (43
antennas of 12 m within 600 m, 20 minutes per galaxy) observing a galaxy at
redshift *z* = 2–3, about 2″ from the phase centre. CO(3–2) (rest 345.796
GHz) falls at 86–115 GHz, in Band 3, where the beam is about 1.4″ × 0.8″.
Each galaxy has

* a CO(3–2) line of 0.4 Jy km/s and 300 km/s FWHM (L′ ≈ 10¹⁰ K km/s pc²: a
  massive star-forming galaxy);
* a dust continuum of 0.2 mJy;
* a size of 0.8″ (FWHM of a circular Gaussian): resolved by the longest
  baselines;
* the primary-beam attenuation at its position;
* noise of 0.1 Jy per visibility (30 s, 20 km/s, close to ALMA in Band 3),
  with weights that misjudge it by `kappa`, channels correlated as after
  Hanning smoothing, and a different depth for every dataset
  (`depth_spread_dex`) — as in real data.

All galaxies have the same luminosity, width, size and centre
(`same_luminosity=True`): the stack, transported to the median redshift
`z_ref`, must return `line_flux`. The same sample is written a second time
**without noise** (`noiseless=True`): the same visibilities and weights, only
the signal.

**Change the parameters** to try other cases: a point source
(`size_arcsec=0`), fewer galaxies or more noise, redshift errors
(`v_jitter_kms`), a spread of luminosities (`flux_scatter_dex`), no
continuum.

In [ ]:
REST_GHZ = 345.79599          # CO(3-2)
SIM = dict(
    n_sources=12,             # galaxies in the stack
    z_range=(2.0, 3.0),       # redshifts, uniform in this range
    line_flux=0.4,            # Jy km/s at z_ref: what the stack should give
    fwhm_kms=300.0,           # line FWHM
    size_arcsec=0.8,          # source FWHM (0 = point source)
    cont_flux=2e-4,           # continuum at the line frequency, Jy (0 = none)
    noise=0.1,                # Jy per visibility (real or imaginary part), typical weight
    kappa=1.3,                # the weights misjudge the noise by this factor
    hanning=True,             # adjacent channels correlated (Hanning smoothing)
    depth_spread_dex=0.2,     # datasets of different depth: noise x 10^(+-0.2)
    v_jitter_kms=0.0,         # redshift errors: random shift of the line centres
    flux_scatter_dex=0.0,     # spread of the luminosities
    same_luminosity=True,
    rmax=600.0,               # array radius, m (beam ~1.4" x 0.8" at 100 GHz)
    seed=0,
)
t0 = time.time()
truth = simulate.make_sample("sim", rest_freq_ghz=REST_GHZ, **SIM)
truth0 = simulate.make_sample("sim_noiseless", rest_freq_ghz=REST_GHZ, noiseless=True, **SIM)
LIST, LIST0 = truth["input_list"], truth0["input_list"]
print(f"2 x {SIM['n_sources']} Measurement Sets in {time.time() - t0:.0f} s, "
      f"z_ref = {truth['z_ref']:.3f}\n")
print(f"{'dataset':16s} {'z':>6s} {'nu_obs':>7s} {'F_line':>7s} {'peak':>9s} {'noise':>6s}")
for s in truth["sources"]:
    print(f"{os.path.basename(s['path']):16s} {s['z']:6.3f} {s['nu_obs_ghz']:7.2f} "
          f"{s['line_flux']:7.3f} {s['line_peak'] * 1e3:5.2f} mJy {s['noise']:6.3f}")

# the true values of the stack, to compare with everything below
T = {"flux": truth["line_flux_stack"], "peak": truth["line_peak_stack"],
     "fwhm": truth["fwhm_kms_stack"], "v0": truth["v0_kms_stack"], "size": SIM["size_arcsec"],
     "cont": truth["cont_flux_stack"]}
print(f"\ntrue stack: line {T['flux']:.3f} Jy km/s, peak {T['peak'] * 1e3:.3f} mJy, "
      f"FWHM {T['fwhm']:.0f} km/s, v0 {T['v0']:+.0f} km/s, size {T['size']}\", "
      f"continuum {T['cont'] * 1e3:.3f} mJy")
print("\nthe input list:\n" + open(LIST).read())

The input list is the plain ViSta one: `<ms_path> <redshift> <RA> <Dec>
[FIELD_ID] [SPW_IDS] [NORM]`, with the position of the galaxy (not of the
phase centre). The galaxies at lower redshift are brighter (they are nearer);
`noise` is the depth of each dataset.

## 2. One galaxy, without and with noise

Before stacking, one galaxy (at the median redshift) measured in the two ways
an interferometer allows, on its noiseless and its noisy dataset.

ViSta works on a single dataset as well: a "stack" of one galaxy shifts its
phase centre to the galaxy, brings it to the rest frame and subtracts the
continuum (`CORRECTED_DATA`); `measure_line` then measures the line flux in
every annulus of baseline length and extrapolates it to zero length. The same
continuum-subtracted visibilities are exported (`export_ms`) and imaged with
CASA's `tclean`: a CLEANed cube, natural weighting.

Some helpers first: the baseline average of a stack, and the image cube with
its spectra (at the peak pixel, in Jy/beam, and summed in an aperture, in
Jy), its moment 0, its beam and the errors of the line flux, measured on the
image itself away from the galaxy.

In [ ]:
line = LineConfig(rest_freq_ghz=REST_GHZ,
                  v_window_kms=(-500, 500))       # the line: excluded from the continuum fit
contsub = ContinuumConfig(order=0,               # a constant per row (1: slope, -1: BIC)
                          exclude_kms=500)        # |v| < 500 km/s left out of the fit
binning = BinningConfig(b_min_klambda=3, b_max_klambda=3000, n_bins=18)
WINDOW = 500                                      # km/s: line flux integrated over |v| < WINDOW


def baseline_average(store, columns=("CORRECTED_DATA",)):
    # every visibility of a stack averaged over the baselines with its weight, per channel
    datasets = xds_from_zarr(store)
    chan_freq = xds_from_zarr(f"{store}::SPECTRAL_WINDOW")[0].CHAN_FREQ.values
    acc = {}
    for ds in datasets:
        freq = np.round(chan_freq[int(ds.DATA_DESC_ID[0])])
        w = ds.WEIGHT.values[:, None, :] * ~ds.FLAG.values
        wsum = w.sum(axis=(0, 2))
        for col in columns:
            num = (w * ds[col].values.real).sum(axis=(0, 2))
            for f, a, b in zip(freq, num, wsum):
                n0, d0 = acc.setdefault(col, {}).get(f, (0.0, 0.0))
                acc[col][f] = (n0 + a, d0 + b)
    freq = np.array(sorted(acc[columns[0]]))
    vel = C_KMS * (line.rest_freq_hz - freq) / line.rest_freq_hz
    return vel, {col: np.array([acc[col][f][0] / acc[col][f][1] for f in freq]) for col in columns}


CELL, NPIX, R_AP = 0.15, 192, 2.0                 # arcsec, pixels, aperture radius (arcsec)


@contextlib.contextmanager
def quiet():
    # tclean writes a progress meter and its log straight to the terminal
    sys.stdout.flush(); sys.stderr.flush()
    saved, null = (os.dup(1), os.dup(2)), os.open(os.devnull, os.O_WRONLY)
    os.dup2(null, 1); os.dup2(null, 2)
    try:
        yield
    finally:
        os.dup2(saved[0], 1); os.dup2(saved[1], 2)
        for fd in (*saved, null):
            os.close(fd)


def image_line(vis, name, noiseless=False):
    # CLEANed cube of the continuum-subtracted visibilities, its spectra and their errors
    for p in glob.glob(name + ".*"):
        shutil.rmtree(p, ignore_errors=True)
    with quiet():
        tclean(vis=vis, imagename=name, datacolumn="corrected", specmode="cube",
               imsize=NPIX, cell=f"{CELL}arcsec", weighting="natural",
               niter=3000 if noiseless else 100000, nsigma=0.0 if noiseless else 2.0,
               threshold="1e-7Jy" if noiseless else "0Jy", usemask="user",
               mask=f"circle[[{NPIX // 2}pix,{NPIX // 2}pix],{3 / CELL:.0f}pix]",
               restoringbeam="common", interpolation="nearest")
    ia = iatool(); ia.open(f"{name}.image")
    cube = ia.getchunk()[:, :, 0, :]                        # (RA, Dec, channel), Jy/beam
    freq = np.array([ia.toworld([0, 0, 0, k])["numeric"][3] for k in range(cube.shape[-1])])
    beam = ia.restoringbeam(); ia.close()
    bmaj, bmin = beam["major"]["value"], beam["minor"]["value"]
    omega = math.pi * bmaj * bmin / (4 * math.log(2)) / CELL ** 2   # beam area, pixels
    ix, iy = np.mgrid[:NPIX, :NPIX]
    dx, dy = (ix - NPIX // 2) * CELL, (iy - NPIX // 2) * CELL     # arcsec from the galaxy
    r = np.hypot(dx, dy)
    vel = C_KMS * (line.rest_freq_hz - freq) / line.rest_freq_hz
    dv = abs(np.median(np.diff(vel)))
    flux_map = cube[:, :, np.abs(vel) <= WINDOW].sum(axis=2) * dv    # Jy/beam km/s
    # the errors, from the image itself away from the galaxy (r > 5"): the rms of the
    # line-flux map (for the peak pixel) and the scatter of the same aperture on empty sky
    sky = [flux_map[np.hypot(dx - a, dy - b) <= R_AP].sum() / omega
           for a in range(-10, 11, 4) for b in range(-10, 11, 4) if math.hypot(a, b) > 5]
    return {"v": vel, "dv": dv, "peak": cube[NPIX // 2, NPIX // 2, :],
            "aperture": cube[r <= R_AP].sum(axis=0) / omega,
            "peak_error": float(np.std(flux_map[r > 5])), "aperture_error": float(np.std(sky)),
            "mom0": cube[:, :, np.abs(vel) <= SIM["fwhm_kms"]].sum(axis=2) * dv,
            "beam": (bmaj, bmin, beam["positionangle"]["value"])}


def integrate(v, spectrum):
    # line flux: the spectrum integrated over |v| < WINDOW
    sel = np.abs(v) <= WINDOW
    return np.nansum(spectrum[sel]) * abs(np.median(np.diff(v)))

In [ ]:
K = SIM["n_sources"] // 2                          # the galaxy at the median redshift
gal, gal0 = truth["sources"][K], truth0["sources"][K]
one = {}
for tag, lst in (("noisy", LIST), ("noiseless", LIST0)):
    entry = [l for l in open(lst) if not l.startswith("#")][K]
    open(f"one_{tag}.txt", "w").write(entry)
    ViSta(f"one_{tag}.txt", verbose=False).run(
        output=f"one_{tag}.zarr", central_freq=line.rest_freq_hz, velocity_range_kms=900,
        line=line, continuum=contsub, stats_line=f"one_{tag}_stats.npy", binning=binning)
    # one galaxy: no redshift rescaling, no primary-beam correction (compared with the image)
    noise = dict(noise="weights", channel_correlation=False) if tag == "noiseless" else {}
    r = measure_line(f"one_{tag}_stats.npy", f"one_{tag}.txt", line,
                     WeightingConfig(redshift_rescaling=False, primary_beam=False),
                     LineFitConfig(mode="detect", v0_kms=0.0, fwhm_kms=SIM["fwhm_kms"],
                                   size="gauss", n_null=0, n_null_phase=0, **noise),
                     BootstrapConfig(n_realisations=0), output_prefix=f"one_{tag}", verbose=False)
    v, avg = baseline_average(f"one_{tag}.zarr")
    export_ms(f"one_{tag}.zarr", f"one_{tag}.ms", verbose=False)
    one[tag] = {"fit": r, "uv": pl.load(f"one_{tag}")["uv"], "v": v,
                "average": avg["CORRECTED_DATA"],
                "image": image_line(f"one_{tag}.ms", f"img_one_{tag}", noiseless=tag == "noiseless")}
print(f"{os.path.basename(gal['path'])}: z = {gal['z']:.3f}, true line flux "
      f"{gal['line_flux'] * gal['primary_beam']:.3f} Jy km/s, size {gal['size_arcsec']}\", "
      f"beam {one['noisy']['image']['beam'][0]:.2f}\" x {one['noisy']['image']['beam'][1]:.2f}\"")

**In the uv plane.** Left: the visibilities averaged over all baselines,
without noise (blue) and with noise (black), and the true total flux density
of the line (green). Right: the line flux in every annulus of baseline
length, with the true curve F·g(b) and the fit of ViSta to the noisy data.

Even without noise the average over the baselines is well below the total:
the long baselines resolve the galaxy and see only part of it. The flux per
annulus shows why — it falls with baseline length, from the total flux at
zero length — and the extrapolation to zero recovers it.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2))
vv = np.linspace(-900, 900, 721)
a1.step(one["noisy"]["v"], one["noisy"]["average"] * 1e3, where="mid", label="with noise", **NOISY)
a1.step(one["noiseless"]["v"], one["noiseless"]["average"] * 1e3, where="mid",
        label="without noise", **CLEAN)
a1.plot(vv, simulate.line_profile(gal, vv) * 1e3, label="true total (zero spacing)", **TRUE)
a1.axhline(0, color="0.5", lw=0.5)
a1.set_xlabel("velocity [km s$^{-1}$]"); a1.set_ylabel("mean Re V over all baselines [mJy]")
a1.legend(loc="upper left")

for tag, sty, mk in (("noisy", NOISY, "o"), ("noiseless", CLEAN, "s")):
    u = one[tag]["uv"]
    ok = (u["used"] == 1) & np.isfinite(u["line_flux"])
    a2.errorbar(u["b_klambda"][ok], u["line_flux"][ok], u["err_thermal"][ok] if tag == "noisy" else None,
                fmt=mk, color=sty["color"], ms=5, capsize=2, mfc="none" if tag == "noiseless" else None,
                label="with noise" if tag == "noisy" else "without noise")
bb = np.geomspace(5, 800, 200)
g = simulate.shape_visibility(bb * 1e3, gal["size_arcsec"])
a2.plot(bb, gal["line_flux"] * gal["primary_beam"] * g, label="true", **TRUE)
ln = one["noisy"]["fit"].summary["line"]
a2.plot(bb, ln["flux"] * simulate.shape_visibility(bb * 1e3, ln["theta_fwhm_arcsec"]),
        color="#c0392b", lw=1.4, label="ViSta fit (with noise)")
a2.axhline(0, color="0.5", lw=0.5)
a2.set_xscale("log"); a2.set_xlabel(r"baseline [k$\lambda$]"); a2.set_ylabel("line flux [Jy km s$^{-1}$]")
lo, hi = a2.get_ylim(); a2.set_ylim(max(lo, -0.5), min(hi, 1.2))
a2.legend()
fig.tight_layout()
save(fig, "one_galaxy_uv")

**In the image.** Top: the line (moment 0 over ±FWHM) without and with
noise, with the true size of the galaxy (green circle, FWHM) and the beam.
Bottom: the spectrum at the peak pixel (Jy/beam) and in an aperture of
radius 2″ (Jy), with the true total.

The peak is in Jy/beam: for a source not much smaller than the beam it is
less than the flux, and its spectrum is, channel by channel, the mean over
the baselines of the uv plane (left, above). The aperture collects the flux
if it is large enough; with noise it also collects the noise of every pixel
in it.

In [ ]:
fig = plt.figure(figsize=(12.5, 8))
ext = NPIX / 2 * CELL
vmax = np.nanmax(one["noiseless"]["image"]["mom0"])
for j, tag in enumerate(("noiseless", "noisy")):
    im_ = one[tag]["image"]
    ax = fig.add_subplot(2, 2, j + 1)
    im = ax.imshow(im_["mom0"].T, origin="lower", extent=(ext, -ext, -ext, ext), cmap="inferno",
                   vmin=-0.3 * vmax, vmax=1.1 * vmax)
    fig.colorbar(im, ax=ax, label="Jy/beam km s$^{-1}$")
    ax.add_patch(Circle((0, 0), gal["size_arcsec"] / 2, fill=False, ec="#2ecc71", lw=1.5, ls="--",
                        label="true size (FWHM)"))
    bmaj, bmin, pa = im_["beam"]
    ax.add_patch(Ellipse((-3.0, -3.0), bmin, bmaj, angle=-pa, fill=False, ec="w", label="beam"))
    ax.add_patch(Circle((0, 0), R_AP, fill=False, ec="w", lw=0.8, ls=":", label=f"aperture {R_AP:g}\""))
    ax.set_xlim(4, -4); ax.set_ylim(-4, 4)
    ax.set_xlabel(r"$\Delta$RA [arcsec]"); ax.set_ylabel(r"$\Delta$Dec [arcsec]")
    ax.set_title("without noise" if tag == "noiseless" else "with noise")
    if j == 0:
        ax.legend(loc="upper left", fontsize=8)
for j, (key, unit) in enumerate((("peak", "Jy/beam"), ("aperture", "Jy"))):
    ax = fig.add_subplot(2, 2, 3 + j)
    for tag, sty in (("noisy", NOISY), ("noiseless", CLEAN)):
        im_ = one[tag]["image"]
        ax.step(im_["v"], im_[key] * 1e3, where="mid",
                label="with noise" if tag == "noisy" else "without noise", **sty)
    ax.plot(vv, simulate.line_profile(gal, vv) * 1e3, label="true total", **TRUE)
    ax.axhline(0, color="0.5", lw=0.5)
    ax.set_xlim(-900, 900)
    ax.set_xlabel("velocity [km s$^{-1}$]")
    ax.set_ylabel(f"{'peak pixel' if key == 'peak' else f'aperture {R_AP:g}″'} [m{unit}]")
    ax.legend(loc="upper left")
fig.tight_layout()
save(fig, "one_galaxy_image")

In [ ]:
def flux_table(rows, title):
    # rows: (label, without noise, with noise, error of the one with noise)
    print(f"{title}\n   {'line flux [Jy km/s]':31s} {'without noise':>13s}   {'with noise':>15s}")
    for lab, a, b, e in rows:
        err = f" ± {e:.3f}" if e is not None else ""
        print(f"   {lab:31s} {a:13.3f}   {b:7.3f}{err}")


u = {tag: one[tag]["fit"].summary["line"] for tag in one}
im = {tag: one[tag]["image"] for tag in one}
F_TRUE = gal["line_flux"] * gal["primary_beam"]
flux_table([
    ("true", F_TRUE, F_TRUE, None),
    ("uv: mean over the baselines", integrate(one["noiseless"]["v"], one["noiseless"]["average"]),
     integrate(one["noisy"]["v"], one["noisy"]["average"]), im["noisy"]["peak_error"]),
    ("uv: ViSta, extrapolated to b=0", u["noiseless"]["flux"], u["noisy"]["flux"],
     u["noisy"]["flux_error"]),
    ("image: peak pixel (Jy/beam)", integrate(im["noiseless"]["v"], im["noiseless"]["peak"]),
     integrate(im["noisy"]["v"], im["noisy"]["peak"]), im["noisy"]["peak_error"]),
    (f"image: aperture {R_AP:g}\"", integrate(im["noiseless"]["v"], im["noiseless"]["aperture"]),
     integrate(im["noisy"]["v"], im["noisy"]["aperture"]), im["noisy"]["aperture_error"]),
], f"{os.path.basename(gal['path'])} (one galaxy)")

Without noise, the flux extrapolated to zero baseline and a large enough
aperture give the true total; the mean over the baselines and the peak pixel
give less, because the long baselines resolve the galaxy. These two are the
same number: every pixel of a naturally weighted image is a weighted mean of
the visibilities, phased to that pixel, and the peak pixel is the mean over
the baselines phased to the galaxy (compare the spectra above).

With noise, every measurement scatters about its noiseless value by about
its error. The errors of the image are measured on the image itself, away
from the galaxy: the rms of the line-flux map for a pixel, and the scatter of
the same aperture on empty sky. The aperture sums the noise of many beams and
has by far the largest error; the uv fit uses every baseline with its weight.
One galaxy is at a few sigma at best: hence the stack.

## 3. Stacking

`ViSta(...).run(...)` reads every dataset, shifts the phase centre to the
galaxy, brings frequencies and baselines to the rest frame, regrids on a
common velocity grid and writes the stack (a Zarr store; a Measurement Set if
the name ends in `.ms`). In the same pass it

* **subtracts the continuum**, row by row, on the channels outside the line
  window: the fit goes to `MODEL_DATA`, the line to `CORRECTED_DATA`, and
  `DATA` is kept (§8 redoes the subtraction);
* **compresses** the stack: for every dataset, channel and annulus of
  baseline length, the sums the measurement needs (`line_stats.npy` from
  `CORRECTED_DATA`, `cont_stats.npy` from `MODEL_DATA`). The measurements
  then take seconds and can be repeated at will.

| Parameter | Meaning |
|---|---|
| `central_freq` | rest-frame centre of the output grid (the line) |
| `velocity_range_kms` | output channels from −X to +X km/s |
| `channel_width_hz` | default: the coarsest rest-frame channel of the sample |
| `LineConfig.v_window_kms` | where the line is: left out of the continuum fit |
| `ContinuumConfig.order` | 0 constant, 1 slope, −1 chosen by the BIC per dataset |
| `BinningConfig` | the annuli: from `b_min` to `b_max` kλ, `n_bins` log-spaced |
| `frame` | `"LSRK"` / `"BARY"` if the input frequencies are TOPO |
| `n_readers`, `n_threads`, `device` | parallelism (`device="gpu"` with a CUDA build) |

Both samples are stacked the same way; the noiseless stack is the reference.

In [ ]:
t0 = time.time()
ViSta(LIST, chunk_rows=5000, verbose=True).run(
    output="stack.zarr",
    central_freq=line.rest_freq_hz,
    velocity_range_kms=900,
    line=line, continuum=contsub,
    stats_line="line_stats.npy", stats_continuum="cont_stats.npy",
    binning=binning)
ViSta(LIST0, verbose=False).run(
    output="stack_noiseless.zarr", central_freq=line.rest_freq_hz, velocity_range_kms=900,
    line=line, continuum=contsub, binning=binning,
    stats_line="line_stats_noiseless.npy", stats_continuum="cont_stats_noiseless.npy")
print(f"\nboth stacked in {time.time() - t0:.0f} s")

**A first look at the stack**: all visibilities of all galaxies averaged
over the baselines, before (`DATA`) and after (`CORRECTED_DATA`) the
continuum subtraction, and the fitted continuum (`MODEL_DATA`), with noise
(solid) and without (dashed). It checks the stacking and the continuum
subtraction, and that the line sits in its window. It is not yet the flux:
resolution, primary beam and redshift rescaling come in §4.

In [ ]:
cols = ("DATA", "MODEL_DATA", "CORRECTED_DATA")
vel, noisy = baseline_average("stack.zarr", cols)
vel0, clean = baseline_average("stack_noiseless.zarr", cols)
fig, ax = plt.subplots(figsize=(8.5, 3.8))
for col, c in zip(cols, ("0.55", "C0", "k")):
    ax.step(vel, noisy[col] * 1e3, where="mid", color=c, lw=1, label=col)
    ax.plot(vel0, clean[col] * 1e3, color=c, ls="--", lw=1.6)
ax.plot([], [], color="0.3", ls="--", label="without noise")
ax.axvspan(*line.v_window_kms, color="C1", alpha=0.08, label="line window")
ax.axhline(0, color="0.5", lw=0.5)
ax.set_xlabel("velocity [km s$^{-1}$]"); ax.set_ylabel("mean Re V over all baselines [mJy]")
lo, hi = ax.get_ylim()
ax.set_ylim(lo, hi + 0.45 * (hi - lo))                 # room for the legend
ax.legend(ncol=3, loc="upper left")
save(fig, "stack_first_look")

## 4. Measuring the stacked line at a known width (`detect`)

`measure_line` fits **all channels and all annuli at once** with

    Re V(channel, b) = F · φ(v; v₀, FWHM) · g(b; θ)

φ a Gaussian line profile, g the visibility of a circular Gaussian source of
FWHM θ (1 for a point source). The noise of every cell is measured from the
scatter of its visibilities (not from the weights), the correlation between
channels from the imaginary part; both enter every error.

* `mode="detect"`: centre and width given (from another line, an optical
  redshift...): a matched filter, the most sensitive measurement.
* `mode="search"` (§5): centre and width free; the significance comes from
  running the same search on noise (`n_null` realisations).
* `size="auto"` (default): a point source and a Gaussian are both fitted and
  averaged with their Akaike weights; the error includes the difference.

The galaxies are transported to `z_ref` (redshift rescaling) and corrected
for the primary beam, so the result compares directly with the true values.
The noiseless stack has no scatter to measure the noise from: it is measured
with the noise of the weights, and its errors mean nothing — only its values.

In [ ]:
weighting = WeightingConfig(scheme="natural", redshift_rescaling=True, primary_beam=True)
boot = BootstrapConfig(n_realisations=200, seed=1)
NOISELESS = dict(noise="weights", channel_correlation=False, n_null=0, n_null_phase=0)


def compare(r, r0=None, name=""):
    # measured (without and with noise) against true, for the main quantities
    def rows(s):
        ln, win = s["line"], s["window"]
        return [("line flux [Jy km/s]", ln["flux"], ln["flux_error"], T["flux"]),
                ("peak [mJy]", ln["peak"] * 1e3, (ln.get("peak_error") or np.nan) * 1e3, T["peak"] * 1e3),
                ("centre v0 [km/s]", ln["v0_kms"], ln.get("v0_error_kms") or np.nan, T["v0"]),
                ("FWHM [km/s]", ln["fwhm_kms"], ln.get("fwhm_error_kms") or np.nan, T["fwhm"]),
                ("size [arcsec]", ln["theta_fwhm_arcsec"],
                 s.get("theta_error_arcsec") or ln.get("theta_error_arcsec") or np.nan, T["size"]),
                ("window flux [Jy km/s]", win["flux"], win["flux_error"], T["flux"])]
    det = r.summary["detection"]
    print(f"{name}   detection S/N {det['snr']:.1f}"
          + (f", {det['significance_sigma']:.1f} sigma" if np.isfinite(det["significance_sigma"]) else ""))
    print(f"   {'':24s} {'true':>8s} {'no noise' if r0 else '':>9s} {'with noise':>17s} {'(m-t)/err':>10s}")
    clean_rows = rows(r0.summary) if r0 else [None] * 6
    for (lab, m, e, t), c in zip(rows(r.summary), clean_rows):
        err = f"± {e:.3g}" if np.isfinite(e) and e > 0 else "(fixed)"
        pull = f"{(m - t) / e:+.1f}" if np.isfinite(e) and e > 0 else ""
        c0 = f"{c[1]:9.3f}" if c else f"{'':9s}"
        print(f"   {lab:24s} {t:8.3f} {c0} {m:8.3f} {err:>8s} {pull:>10s}")


det = measure_line("line_stats.npy", LIST, line, weighting,
                   LineFitConfig(mode="detect", v0_kms=0.0, fwhm_kms=SIM["fwhm_kms"],
                                 n_null=500, n_null_phase=100),
                   boot, output_prefix="detect")
det0 = measure_line("line_stats_noiseless.npy", LIST0, line, weighting,
                    LineFitConfig(mode="detect", v0_kms=0.0, fwhm_kms=SIM["fwhm_kms"], **NOISELESS),
                    BootstrapConfig(n_realisations=0), output_prefix="detect_noiseless", verbose=False)
print()
compare(det, det0, "detect")

`measure_line` wrote `detect_results.json` (every number), `_profile.txt`,
`_uvamp.txt`, `_noise.txt` and `_diagnostics.npz`; `examples/plot_line.py`
draws the figures from them, with the true signal as a `reference` curve,
and the noiseless measurement is added on top.

* **Profile**: the flux density at zero spacing in every channel (the total
  flux density of the stack), the fit, ±1σ noise, the imaginary part (a null
  test: noise only) and the residuals.
* **Line flux per annulus** against baseline length: flat for a point
  source, falling for a resolved one; its value at zero is the total flux.
* **Noise**: per channel from the scatter of the visibilities and from the
  weights (their ratio, κ, was 1.3 in the simulation), the standardised cells
  against N(0,1), and the null distribution of the detection S/N.

In [ ]:
V_TRUE = np.linspace(-900, 900, 721)
B_TRUE = np.geomspace(5, 800, 200)
REF_PROFILE = (V_TRUE, simulate.true_profile(truth, V_TRUE))
REF_UV = (B_TRUE, simulate.true_uv(truth, B_TRUE))


def add_noiseless(fig, d0, kind):
    # the noiseless measurement on top of a plot_line figure
    f, _ = pl.flux_scale("Jy", d0)
    ax = fig.axes[0]
    if kind == "profile":
        p = d0["prof"]
        o = np.argsort(p["v_kms"])
        ax.step(p["v_kms"][o], p["flux_density"][o] * f, where="mid", label="without noise",
                **dict(CLEAN, lw=2.2, alpha=0.8, zorder=1.5))      # under the true curve
    else:
        u = d0["uv"]
        ok = (u["used"] == 1) & np.isfinite(u["line_flux"])
        ax.plot(u["b_klambda"][ok], u["line_flux"][ok] * f, "s", mfc="none", color=CLEAN["color"],
                ms=6, label="without noise")
    ax.legend(loc="upper right")
    return fig


d, d0 = pl.load("detect"), pl.load("detect_noiseless")
save(add_noiseless(pl.plot_profile(d, unit="Jy", reference=REF_PROFILE), d0, "profile"),
     "detect_profile")
save(add_noiseless(pl.plot_uv(d, unit="Jy", reference=REF_UV), d0, "uv"), "detect_uv")
save(pl.plot_noise(d, unit="Jy"), "detect_noise")

In [ ]:
n = det.summary["noise"]
print(f"noise / weights (kappa):  measured {n['kappa_scatter_over_weights']:.3f}   true {SIM['kappa']}")
rho = n["channel_correlation"]
print(f"channel correlation:      measured {rho[0]:+.2f} {rho[1]:+.2f} {rho[2]:+.2f}   "
      f"true (Hanning) {'+0.67 +0.17 +0.00' if SIM['hanning'] else '0 0 0'}")
print(f"imaginary rms / noise:    {n['imaginary_rms_over_noise']:.3f}   (1 = noise only)")
print(f"line-free rms / noise:    {n['line_free_rms_over_noise']:.3f}   (1 = no residual continuum)")
st = det.summary["size_test"]
print(f"size test: chi2(point) - chi2(Gaussian) = {st['delta_chi2_point_minus_gauss']:.1f} "
      f"(correlation factor {st['correlation_factor']:.2f}): adopted {st['adopted']}, "
      f"Gaussian weight {st['weight_gauss']:.2f}")

Without noise the stack returns the true values: the stacking, the
rescaling, the primary-beam correction and the fit add no bias. With noise
the measurement scatters around them by about its error. The noise from the
scatter recovers the factor by which the weights are wrong, and the channel
correlation that of Hanning smoothing; the imaginary part and the line-free
channels are at the noise level.

## 5. Searching for the line (`search`)

Centre and width free, over the velocity window. The significance accounts
for having looked at many centres, widths and sizes: it is the probability
that noise alone gives a peak as high, from `n_null` noise realisations with
the measured noise and correlation (and, with `n_null_phase`, from stacks
with random phases).

In [ ]:
search = measure_line("line_stats.npy", LIST, line, weighting,
                      LineFitConfig(mode="search", n_null=500, n_null_phase=100),
                      boot, output_prefix="search")
search0 = measure_line("line_stats_noiseless.npy", LIST0, line, weighting,
                       LineFitConfig(mode="search", **NOISELESS),
                       BootstrapConfig(n_realisations=0), output_prefix="search_noiseless",
                       verbose=False)
print()
compare(search, search0, "search")
save(add_noiseless(pl.plot_profile(pl.load("search"), unit="Jy", rebin=2, reference=REF_PROFILE),
                   pl.load("search_noiseless"), "profile"), "search_profile")

## 6. What each option changes

The statistics are a few MB: every variant takes seconds. Each one changes
**one** thing with respect to the `detect` measurement of §4.

| Option | What it does | When |
|---|---|---|
| `size="point"` | point source only | the source is known to be unresolved |
| `size="gauss"` | Gaussian only | clearly resolved |
| `theta_prior_arcsec=(θ, σ_dex)` | soft prior on the size | a weak line, with a size from the continuum or another line |
| `theta_fixed_arcsec=θ` | size fixed | the same, a hard version |
| `noise="weights"` | noise from the weights, not the scatter | weights known to be absolute |
| `channel_correlation=False` | channels treated as independent | (shown to see the effect) |
| `scheme="democratic"` | every galaxy counts the same | the mean of the galaxies rather than of the data |
| `redshift_rescaling=False` | no transport to z_ref | a narrow redshift range |
| `primary_beam=False` | no primary-beam correction | data already corrected |
| `fwhm_kms` in `detect` | the assumed width | |

In [ ]:
def variant(weighting=weighting, **fit):
    cfg = dict(mode="detect", v0_kms=0.0, fwhm_kms=SIM["fwhm_kms"], n_null=200, n_null_phase=0)
    cfg.update(fit)
    return measure_line("line_stats.npy", LIST, line, weighting, LineFitConfig(**cfg),
                        BootstrapConfig(n_realisations=100, seed=1), verbose=False)


variants = {
    "reference (size auto)": variant(),
    "size = point": variant(size="point"),
    "size = gauss": variant(size="gauss"),
    "size prior 0.8\" ± 0.15 dex": variant(theta_prior_arcsec=(0.8, 0.15)),
    "size fixed 0.8\"": variant(theta_fixed_arcsec=0.8),
    "noise from the weights": variant(noise="weights"),
    "no channel correlation": variant(channel_correlation=False),
    "democratic weighting": variant(WeightingConfig(scheme="democratic")),
    "no redshift rescaling": variant(WeightingConfig(redshift_rescaling=False)),
    "no primary-beam correction": variant(WeightingConfig(primary_beam=False)),
    "assumed FWHM 200 km/s": variant(fwhm_kms=200.0),
    "assumed FWHM 400 km/s": variant(fwhm_kms=400.0),
}

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(13, 5.2), sharey=True,
                                 gridspec_kw={"width_ratios": [1.3, 1, 0.8]})
names = list(variants)
y = np.arange(len(names))[::-1]
for yi, name in zip(y, names):
    s = variants[name].summary
    ln = s["line"]
    a1.errorbar(ln["flux"], yi, xerr=ln["flux_error"], fmt="o", color="k", ms=5, capsize=2)
    te = s.get("theta_error_arcsec") or ln.get("theta_error_arcsec") or 0
    a2.errorbar(ln["theta_fwhm_arcsec"], yi, xerr=te, fmt="o", color="k", ms=5, capsize=2)
    a3.barh(yi, s["detection"]["snr"], color="0.6", height=0.6)
a1.axvline(T["flux"], label="true signal", **TRUE)
a2.axvline(T["size"], **TRUE)
a1.set_yticks(y); a1.set_yticklabels(names)
a1.set_xlabel("line flux [Jy km s$^{-1}$]"); a2.set_xlabel("size [arcsec]")
a3.set_xlabel("detection S/N")
a1.legend()
fig.tight_layout()
save(fig, "options")

What the figure shows (the numbers are those of the default simulation):

* a **point-source model** keeps only part of the flux of this 0.8″ source —
  any extended emission is lost the same way. A size prior or a fixed size
  helps when the line is weak and the size known from elsewhere;
* with the **noise from the weights** (wrong by κ = 1.3) the flux error is
  rescaled by √χ²ᵣ and survives, but the detection S/N is overstated by κ —
  and ViSta warns that the imaginary part and the line-free channels scatter
  more than that noise;
* **without the channel correlation** the errors are too small and the S/N
  much too high: never on Hanning-smoothed data;
* **democratic** weighting gives the shallow datasets the weight of the deep
  ones: a larger error;
* **without the redshift rescaling** the stack is the mean of galaxies at
  different distances, not the flux at `z_ref`; **without the primary-beam
  correction** it is a little low;
* a **wrong width** in `detect` mode biases the flux: when the width is not
  known, use `search` (§5).

## 7. The continuum

The continuum removed during the stacking (`MODEL_DATA`) was compressed into
`cont_stats.npy`. `extract_flux` with `method="continuum"` averages it over
the line-free channels and fits its uv profile: flux at zero spacing and
size.

In [ ]:
conts = {}
for tag, stats, lst in (("noisy", "cont_stats.npy", LIST),
                        ("noiseless", "cont_stats_noiseless.npy", LIST0)):
    conts[tag] = extract_flux(stats, lst, line, weighting, profile=ProfileConfig(method="continuum"),
                              uvfit=UVFitConfig(model="gauss"), bootstrap=boot,
                              output_prefix=f"continuum_{tag}", verbose=False)
cs, cs0 = conts["noisy"].summary, conts["noiseless"].summary
print(f"continuum flux [mJy]   true {T['cont'] * 1e3:.3f}   without noise {cs0['flux_total'] * 1e3:.3f}"
      f"   with noise {cs['flux_total'] * 1e3:.3f} ± {cs['flux_total_error'] * 1e3:.3f}")
print(f"continuum size [\"]     true {T['size']:.2f}    without noise {cs0['theta_fwhm_arcsec']:.2f}"
      f"    with noise {cs['theta_fwhm_arcsec']:.2f} ± {cs['theta_error_arcsec']:.2f}")

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for tag, mk in (("noisy", "o"), ("noiseless", "s")):
    c = conts[tag]
    ok = np.isfinite(c.flux) & c.bin_ok
    ax.errorbar(c.baseline_klambda[ok], c.flux[ok] * 1e3, c.flux_error[ok] * 1e3 if tag == "noisy" else None,
                fmt=mk, color=(NOISY if tag == "noisy" else CLEAN)["color"], ms=5, capsize=2,
                mfc="none" if tag == "noiseless" else None,
                label="with noise" if tag == "noisy" else "without noise")
g = simulate.shape_visibility(B_TRUE * 1e3, cs["theta_fwhm_arcsec"])
ax.plot(B_TRUE, cs["flux_total"] * g * 1e3, color="#c0392b", lw=1.4, label="fit (with noise)")
ax.plot(B_TRUE, simulate.true_uv(truth, B_TRUE, "continuum") * 1e3, label="true signal", **TRUE)
ax.axhline(0, color="0.5", lw=0.5)
ax.set_xscale("log"); ax.set_xlabel(r"baseline [k$\lambda$]"); ax.set_ylabel("continuum [mJy]")
ax.legend()
save(fig, "continuum_uv")

## 8. Redoing the continuum subtraction

`DATA` stays in the stack, so the subtraction can be redone — for instance
if the first look of §3 shows a feature in the line-free channels, or to see
how much the result depends on the window and the order — and the stack
compressed and measured again, without stacking again. Every subtraction is
recorded in the stack and in the results (`contsub`). Here: a slope instead
of a constant, outside ±600 km/s instead of ±500.

In [ ]:
line2 = LineConfig(rest_freq_ghz=REST_GHZ, v_window_kms=(-600, 600))
subtract_continuum("stack.zarr", line2, ContinuumConfig(order=1, exclude_kms=600), verbose=False)
compress_visibilities("stack.zarr", LIST, line2, binning=binning, line_column="CORRECTED_DATA",
                      out_line="line_stats_v2.npy", out_continuum="cont_stats_v2.npy",
                      resume=False, verbose=False)
redo = measure_line("line_stats_v2.npy", LIST, line2, weighting,
                    LineFitConfig(mode="detect", v0_kms=0.0, fwhm_kms=SIM["fwhm_kms"], n_null=200,
                                  n_null_phase=0), boot, output_prefix="redo", verbose=False)
compare(redo, None, "order 1, outside ±600 km/s")
rec = redo.summary["contsub"]
print("\nrecorded:", {k: rec[k] for k in ("mode", "order", "v_window_kms", "exclude_kms")})
save(pl.plot_profile(pl.load("redo"), unit="Jy", reference=REF_PROFILE), "redo_profile")

A slope fitted on fewer line-free channels adds noise to every channel, so
the error grows; with a flat continuum, as here, order 0 is the better
choice. `order=-1` lets the BIC choose for each dataset.

## 9. The stack in the image, without and with noise

The stack exported as a Measurement Set (`export_ms`) and imaged as in §2.
The stacked visibilities are the plain weighted mean of the galaxies: no
redshift rescaling and no primary-beam correction, which ViSta applies only
when it measures. For a like-for-like comparison, ViSta is run here without
them too. The true signal is the noiseless stack: the same visibilities
without noise.

In [ ]:
stack_img, plain = {}, {}
for tag, store, stats, lst in (("noisy", "stack.zarr", "line_stats.npy", LIST),
                               ("noiseless", "stack_noiseless.zarr", "line_stats_noiseless.npy", LIST0)):
    export_ms(store, f"{store[:-5]}.ms", verbose=False)
    stack_img[tag] = image_line(f"{store[:-5]}.ms", f"img_{tag}", noiseless=tag == "noiseless")
    extra = NOISELESS if tag == "noiseless" else dict(n_null=0, n_null_phase=0)
    plain[tag] = measure_line(stats, lst, line, WeightingConfig(redshift_rescaling=False, primary_beam=False),
                              LineFitConfig(mode="detect", v0_kms=0.0, fwhm_kms=SIM["fwhm_kms"], **extra),
                              BootstrapConfig(n_realisations=0 if tag == "noiseless" else 100),
                              verbose=False).summary["line"]

fig = plt.figure(figsize=(12.5, 8))
vmax = np.nanmax(stack_img["noiseless"]["mom0"])
for j, tag in enumerate(("noiseless", "noisy")):
    im_ = stack_img[tag]
    ax = fig.add_subplot(2, 2, j + 1)
    im = ax.imshow(im_["mom0"].T, origin="lower", extent=(ext, -ext, -ext, ext), cmap="inferno",
                   vmin=-0.3 * vmax, vmax=1.1 * vmax)
    fig.colorbar(im, ax=ax, label="Jy/beam km s$^{-1}$")
    ax.add_patch(Circle((0, 0), SIM["size_arcsec"] / 2, fill=False, ec="#2ecc71", lw=1.5, ls="--",
                        label="true size (FWHM)"))
    bmaj, bmin, pa = im_["beam"]
    ax.add_patch(Ellipse((-3.0, -3.0), bmin, bmaj, angle=-pa, fill=False, ec="w", label="beam"))
    ax.add_patch(Circle((0, 0), R_AP, fill=False, ec="w", lw=0.8, ls=":", label=f"aperture {R_AP:g}\""))
    ax.set_xlim(4, -4); ax.set_ylim(-4, 4)
    ax.set_xlabel(r"$\Delta$RA [arcsec]"); ax.set_ylabel(r"$\Delta$Dec [arcsec]")
    ax.set_title(("without noise" if tag == "noiseless" else "with noise") + " (stack)")
    if j == 0:
        ax.legend(loc="upper left", fontsize=8)
for j, (key, unit) in enumerate((("peak", "Jy/beam"), ("aperture", "Jy"))):
    ax = fig.add_subplot(2, 2, 3 + j)
    for tag, sty in (("noisy", NOISY), ("noiseless", CLEAN)):
        im_ = stack_img[tag]
        ax.step(im_["v"], im_[key] * 1e3, where="mid",
                label="with noise" if tag == "noisy" else "without noise (the true stack)", **sty)
    ax.axhline(0, color="0.5", lw=0.5)
    ax.set_xlim(-900, 900)
    ax.set_xlabel("velocity [km s$^{-1}$]")
    ax.set_ylabel(f"{'peak pixel' if key == 'peak' else f'aperture {R_AP:g}″'} [m{unit}]")
    ax.legend(loc="upper left")
fig.tight_layout()
save(fig, "stack_image")

im = stack_img
avg = {tag: baseline_average(store) for tag, store in (("noisy", "stack.zarr"),
                                                       ("noiseless", "stack_noiseless.zarr"))}
flux_table([
    ("uv: mean over the baselines", integrate(avg["noiseless"][0], avg["noiseless"][1]["CORRECTED_DATA"]),
     integrate(avg["noisy"][0], avg["noisy"][1]["CORRECTED_DATA"]), im["noisy"]["peak_error"]),
    ("uv: ViSta, extrapolated to b=0", plain["noiseless"]["flux"],
     plain["noisy"]["flux"], plain["noisy"]["flux_error"]),
    ("image: peak pixel (Jy/beam)", integrate(im["noiseless"]["v"], im["noiseless"]["peak"]),
     integrate(im["noisy"]["v"], im["noisy"]["peak"]), im["noisy"]["peak_error"]),
    (f"image: aperture {R_AP:g}\"", integrate(im["noiseless"]["v"], im["noiseless"]["aperture"]),
     integrate(im["noisy"]["v"], im["noisy"]["aperture"]), im["noisy"]["aperture_error"]),
], "the stack, no rescaling (without noise = the true stack)")

As for one galaxy: the mean over the baselines and the peak pixel are the
same number, below the total because the stack is resolved; the flux
extrapolated to zero baseline and the aperture recover the total of the
noiseless stack. With noise, the uv fit and the aperture find about the same
flux, but the error of the aperture is several times larger: it sums the
noise of every pixel in it, while the fit weighs every baseline by what it
knows about the source.

The image shows where the emission is and whether it is symmetric, which the
uv fit assumes: the two are complementary. For the flux of a resolved stack
the uv measurement needs no aperture and no deconvolution.

## 10. From the command line, and on your own data

The shell driver of the repository runs the same steps (the package must be
importable: `PYTHONPATH` points to the repository). It stacks again, so it
takes as long as §3–4; the results go to `cli/`:

In [ ]:
cmd = (f"PYTHONPATH={REPO} bash {REPO}/run_vista.sh --input {LIST} --rest-freq {REST_GHZ} "
       f"--v-window '-500 500' --velocity-range 900 --contsub before "
       f"--method joint --mode detect --fwhm {SIM['fwhm_kms']:g} --unit Jy --out-dir cli")
print(cmd, "\n")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=WORK)
log = r.stdout + r.stderr
print("\n".join(l for l in log.splitlines() if l.startswith("==") or "Line:" in l)
      if r.returncode == 0 else log[-3000:])
print(f"\ntrue line flux {T['flux']:.4f}")

On your own data:

1. write the input list, one line per dataset:
   `<ms_path> <redshift> <RA> <Dec> [FIELD_ID] [SPW_IDS] [NORM]`, with the
   position of the galaxy (hh:mm:ss.s ±dd:mm:ss.s);
2. stack with the continuum subtraction and the compression in one pass
   (§3), with a `v_window_kms` wide enough to contain the whole line;
3. look at the stack (§3) and at `line_free_rms_over_noise`; if needed,
   redo the subtraction (§8);
4. measure (§4–5): `detect` if the width is known, otherwise `search` and
   its `significance_sigma`; check `imaginary_rms_over_noise`, the size test
   and the ratio of bootstrap to thermal error (much larger than 1: the
   galaxies differ more than the noise);
5. draw the figures with `examples/plot_line.py`: colours, lines and fonts
   are in its `STYLE` dictionary (or a matplotlib style file,
   `pl.apply_style("my.mplstyle")`).

`./run_vista.sh --help` lists every option; `README.md` and
`docs/extraction.md` explain them.

In [ ]:
# the files written in this run
for f in sorted(os.listdir(WORK)):
    if not os.path.isdir(f):
        print(f"{f:36s} {os.path.getsize(f) / 1e3:10.1f} kB")